# 02 — Operators on Function Spaces and the Laplacian

**The question:** what is a linear operator on a function space, and what makes the Laplace-Beltrami operator the central object of spectral geometry?

## Intuition

In linear algebra, a matrix $A \in \mathbb{R}^{n \times n}$ acts on vectors: it takes a vector $\mathbf{v}$ and returns another vector $A\mathbf{v}$. An **operator** does the same thing, but for functions: it takes a function $f$ defined on a shape and returns another function $Af$.

The most important operator in spectral geometry is the **Laplace-Beltrami operator (LBO)**, $\Delta$. It generalises the familiar scalar Laplacian $\Delta f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$ from flat 2D to curved surfaces. Its eigenvalue problem — *"which functions does $\Delta$ map to a scaled copy of themselves?"* — defines the spectral basis we will use throughout this series.

## Minimal math

### Linear operators on $L^2(M)$

A **linear operator** $A : L^2(M) \to L^2(M)$ satisfies:

$$A(\alpha f + \beta g) = \alpha\, Af + \beta\, Ag \quad \forall\, f, g \in L^2(M),\; \alpha, \beta \in \mathbb{R}$$

An operator is **self-adjoint** with respect to the $L^2(M)$ inner product if:

$$\langle Af, g \rangle_M = \langle f, Ag \rangle_M \quad \forall\, f, g \in L^2(M)$$

Self-adjoint operators have two fundamental properties:
1. All **eigenvalues are real**
2. Eigenfunctions belonging to different eigenvalues are **orthogonal** in $L^2(M)$

These properties make self-adjoint operators the foundation of spectral analysis.

### The Laplace-Beltrami operator

The **LBO** is defined as:

$$\Delta f = -\mathrm{div}(\nabla f)$$

i.e. the divergence of the gradient of $f$, both taken intrinsically on the surface. It is self-adjoint and **positive semi-definite** with respect to $\langle \cdot, \cdot \rangle_M$:

$$\langle \Delta f, f \rangle_M = \int_M \|\nabla f\|^2 \, dA \geq 0$$

The right-hand side is exactly the **Dirichlet energy** $E_D(f)$ from notebook 03 — so $\Delta$ measures roughness.

### Discrete LBO: stiffness and mass matrices

On a triangle mesh, $\Delta$ is discretised as:

$$\Delta_M \mathbf{f} = M^{-1} S\, \mathbf{f}$$

where:
- $S$ is the **cotangent stiffness matrix**: $S_{ij} = -\frac{1}{2}(\cot\alpha_{ij} + \cot\beta_{ij})$ for adjacent vertices $i,j$
- $M = \mathrm{diag}(a_1,\ldots,a_n)$ is the **mass matrix** from notebook 01

The discrete Dirichlet energy is $E_D(\mathbf{f}) = \mathbf{f}^\top S\, \mathbf{f}$, and the self-adjointness condition becomes $\mathbf{f}^\top S\, \mathbf{g} = \mathbf{g}^\top S\, \mathbf{f}$ (which holds because $S$ is symmetric).

### The generalised eigenvalue problem

The eigenfunctions of $\Delta_M$ satisfy:

$$\Delta_M \phi_k = \lambda_k \phi_k \quad \Longleftrightarrow \quad S\, \phi_k = \lambda_k M\, \phi_k$$

with:

$$0 = \lambda_0 \leq \lambda_1 \leq \lambda_2 \leq \cdots, \qquad \phi_i^\top M\, \phi_j = \delta_{ij}$$

These eigenfunctions form the **Laplace-Beltrami spectral basis** — the subject of notebook 03.

In [1]:
import gsops.backend as gs
import numpy as np
import polyscope as ps

import geomfum.linalg as la
from geomfum.dataset import NotebooksDataset
from geomfum.shape import TriangleMesh

ps.init()

In [2]:
dataset = NotebooksDataset()
mesh = TriangleMesh.from_file(dataset.get_filename("cat-00"))

V = mesh.vertices
F = mesh.faces

## The stiffness and mass matrices

GeomFuM computes both matrices on demand. They are sparse — on a mesh with $n$ vertices, each vertex is connected to $\approx 6$ neighbours, so $S$ has $O(n)$ non-zero entries.

In [3]:
S = mesh.laplacian.stiffness_matrix  # cotangent stiffness, sparse (n, n)
M = mesh.laplacian.mass_matrix  # diagonal mass, sparse (n, n)

n = mesh.n_vertices
print(f"Stiffness S: shape {S.shape}, non-zeros {S.nnz} = {S.nnz / n:.1f} per row")
print(f"Mass      M: shape {M.shape}, diagonal")
print()

# S is symmetric (necessary for self-adjointness)
diff = S - S.T
print(f"||S - S^T||_max = {np.max(np.abs(diff)):.2e}  (should be ~0 — S is symmetric)")

Stiffness S: shape (7207, 7207), non-zeros 50437 = 7.0 per row
Mass      M: shape (7207, 7207), diagonal

||S - S^T||_max = 0.00e+00  (should be ~0 — S is symmetric)


## Applying the Laplacian to a function

`mesh.laplacian(f)` computes $\Delta_M \mathbf{f} = M^{-1} S \mathbf{f}$ natively.

In [4]:
# A constant function: Δ(constant) = 0  (constants are in the kernel)
f_const = gs.ones(n)
Df_const = mesh.laplacian(f_const)
print(f"max |Δ(1)|  = {float(max(gs.abs(Df_const))):.2e}   (should be ~0)")

# The height function: Δ(z) measures how much z deviates from its local average
f_z = gs.from_numpy(V[:, 2])
Df_z = mesh.laplacian(f_z)
print(f"max |Δ(f_z)| = {float(max(gs.abs(Df_z))):.4f}")

max |Δ(1)|  = 1.21e-08   (should be ~0)
max |Δ(f_z)| = 6704.5233


In [5]:
# Visualise f_z and Δ(f_z) side by side
ps.remove_all_structures()
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)
ps_cat.add_scalar_quantity("f_z", (f_z), enabled=False, cmap="coolwarm")
ps_cat.add_scalar_quantity("Δ(f_z)", (Df_z), enabled=True, cmap="coolwarm")

ps.show()
# Δ(f_z) is large where f_z deviates most from its neighbourhood average —
# e.g. at high-curvature regions like the ears and paws.

## Self-adjointness: $\langle \Delta f, g \rangle_M = \langle f, \Delta g \rangle_M$

This is the discrete statement that $S$ is symmetric. We verify it numerically.

In [6]:
f = gs.from_numpy(V[:, 0])  # x-coordinate
g = gs.from_numpy(V[:, 2])  # z-coordinate

Df = mesh.laplacian(f)
Dg = mesh.laplacian(g)

# <Δf, g>_M  =  (Δf)^T M g
lhs = float(Df @ la.matvecmul(M, g))
# <f, Δg>_M  =  f^T M (Δg)
rhs = float(f @ la.matvecmul(M, Dg))

print(f"<Δf, g>_M  = {lhs:.8f}")
print(f"<f,  Δg>_M = {rhs:.8f}")
print(f"Difference = {abs(lhs - rhs):.2e}  (should be ~0)")

<Δf, g>_M  = -0.00078535
<f,  Δg>_M = -0.00078535
Difference = 1.09e-16  (should be ~0)


## Dirichlet energy: $E_D(f) = \langle \Delta f, f \rangle_M = \mathbf{f}^\top S\, \mathbf{f}$

The Dirichlet energy is the inner product of $f$ with its own Laplacian. Since $\langle \Delta f, f \rangle_M = \mathbf{f}^\top S\, \mathbf{f} \geq 0$, the LBO is **positive semi-definite**.

In [7]:
def dirichlet(f_gs):
    """E_D(f) = f^T S f  (= <Δf, f>_M for self-adjoint Δ)."""
    Sf = la.matvecmul(S, f_gs)
    return float(f_gs @ Sf)


f_const_gs = gs.ones(n)
f_sin_gs = gs.from_numpy(np.sin(8 * np.pi * V[:, 2] / (V[:, 2].max() - V[:, 2].min())))

print(
    f"E_D(constant) = {dirichlet(f_const_gs):.6f}   (zero — constants have zero gradient)"
)
print(f"E_D(f_z)      = {dirichlet(f_z):.4f}")
print(f"E_D(f_sin)    = {dirichlet(f_sin_gs):.4f}   (larger — f_sin oscillates more)")

E_D(constant) = -0.000000   (zero — constants have zero gradient)
E_D(f_z)      = 0.2863
E_D(f_sin)    = 135.4205   (larger — f_sin oscillates more)


## The eigenvalue problem

The LBO eigenfunctions satisfy $\Delta_M \phi_k = \lambda_k \phi_k$. We compute a few and verify the equation numerically.

In [8]:
K = 20
mesh.laplacian.find_spectrum(spectrum_size=K, set_as_basis=True)

lam = mesh.basis.vals  # (K,) eigenvalues
Phi = mesh.basis.vecs  # (n, K) eigenvectors

print(f"First {K} eigenvalues:")
print(lam.round(4))

First 20 eigenvalues:
[  0.      17.9035  34.0199  53.0269  66.778   68.4836  88.1972 139.0265
 215.9279 216.5246 219.1368 280.9364 297.8642 311.9572 359.8241 412.8476
 438.7461 445.5958 501.7155 517.2657]


In [9]:
# Verify:  Δ φ_k  ≈  λ_k φ_k
for k in [1, 5, 10, 19]:
    phi_k = Phi[:, k]
    lhs = mesh.laplacian(phi_k)  # Δ φ_k
    rhs = lam[k] * phi_k  # λ_k φ_k
    res = float(max(gs.abs(lhs - rhs)))
    print(f"  k={k:2d}  λ_k={lam[k]:.4f}  max|Δφ_k - λ_k φ_k| = {res:.2e}")

  k= 1  λ_k=17.9035  max|Δφ_k - λ_k φ_k| = 7.51e-08
  k= 5  λ_k=68.4836  max|Δφ_k - λ_k φ_k| = 2.08e-07
  k=10  λ_k=219.1368  max|Δφ_k - λ_k φ_k| = 1.48e-07
  k=19  λ_k=517.2657  max|Δφ_k - λ_k φ_k| = 1.64e-07


In [10]:
Phi.shape

(7207, 20)

In [11]:
# Check M-orthonormality: Φ^T M Φ = I
MPhi = la.matvecmul(M, Phi.T).T  # M Φ,  shape (n, K)
gram = Phi.T @ MPhi  # Φ^T M Φ,  should be identity

off_diag_error = np.max(np.abs(gram - np.eye(K)))
print(f"max |Φ^T M Φ - I| = {off_diag_error:.2e}  (should be ~0 — M-orthonormal)")

max |Φ^T M Φ - I| = 1.11e-15  (should be ~0 — M-orthonormal)


## Visualise: the Laplacian applied to an eigenfunction

In [12]:
k_show = 5  # try different k
phi_k = Phi[:, k_show]
Dphi_k = mesh.laplacian(gs.from_numpy(phi_k))

ps.remove_all_structures()
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)
ps_cat.add_scalar_quantity(f"phi_{k_show}", phi_k, enabled=True, cmap="coolwarm")
ps_cat.add_scalar_quantity(
    f"Delta(phi_{k_show})", Dphi_k, enabled=False, cmap="coolwarm"
)
ps_cat.add_scalar_quantity(
    f"lambda*phi_{k_show}", lam[k_show] * phi_k, enabled=False, cmap="coolwarm"
)

print(f"λ_{k_show} = {lam[k_show]:.4f}")
print("Enable 'Delta(phi)' and 'lambda*phi' in polyscope — they should look identical.")
ps.show()

λ_5 = 68.4836
Enable 'Delta(phi)' and 'lambda*phi' in polyscope — they should look identical.


## Where to go next

- [03 — What is a Basis?](./03_what_is_a_basis.ipynb): the eigenfunctions of $\Delta_M$ form the spectral basis
- [05 — The Laplacian and Frequency](./05_the_laplacian_and_frequency.ipynb): eigenvalues as intrinsic frequencies